# Intent Gap — Stage A pilot (v1)

Stage A regex filter on a streamed sample of WildChat-1M (Zhao et al., 2024). Surfaces conversations where user turn 2 contains a conversational-repair signal, as a label-free proxy for intent-gap failures.

In [ ]:
!pip install -q datasets

## Repair-signal filter (v1 seed list)

17 phrases derived from the academic dialogue-breakdown / conversational-repair literature.

In [ ]:
import re

REPAIR_PHRASES = [
    r"no,?\s+i\s+meant",
    r"that(?:'s|\s+is)\s+not\s+what\s+i\s+(?:asked|wanted|meant)",
    r"you\s+misunderstood",
    r"you\s+missed\s+the\s+point",
    r"let\s+me\s+rephrase",
    r"let\s+me\s+clarify",
    r"to\s+be\s+clear,?\s+i\s+(?:want|meant|need)",
    r"actually,?\s+i\s+(?:want|meant|need)",
    r"what\s+i\s+actually\s+(?:need|want|meant)",
    r"the\s+question\s+was",
    r"you\s+didn'?t\s+answer\s+my\s+question",
    r"read\s+the\s+prompt\s+again",
    r"this\s+is\s+(?:wrong|incorrect|not\s+what\s+i'?m\s+looking\s+for)",
    r"you\s+got\s+it\s+wrong",
    r"(?:not\s+useful|useless|unhelpful)",
    r"i\s+think\s+you\s+misunderstood",
    r"that\s+wasn'?t\s+the\s+question",
]
REPAIR_RE = re.compile('|'.join(f'({p})' for p in REPAIR_PHRASES), flags=re.IGNORECASE)


def has_repair_signal(text: str) -> bool:
    return bool(REPAIR_RE.search(text or ''))

## WildChat-1M stream

Streaming load avoids the ~25 GB full-corpus download. Restrict to English; the non-toxic subset is used to skip moderation-flagged conversations.

In [ ]:
from datasets import load_dataset

ds = load_dataset('allenai/WildChat-1M', split='train', streaming=True)

## Stage A filter — 10,000-conversation sample

Inclusion criteria: English, $\geq$ 4 conversation turns (so that a repair turn is at least possible), repair signal on user turn 2.

In [ ]:
import json
from collections import Counter

SAMPLE_SIZE = 10_000

n_total = 0
n_english = 0
n_long_enough = 0
n_repair_signal = 0
phrase_counter = Counter()
candidates = []

for row in ds:
    if n_total >= SAMPLE_SIZE:
        break
    n_total += 1
    if row.get('language', '').lower() != 'english':
        continue
    n_english += 1
    turns = row.get('conversation', [])
    if len(turns) < 4:
        continue
    n_long_enough += 1
    try:
        user_t1 = turns[0]['content']
        asst_t1 = turns[1]['content']
        user_t2 = turns[2]['content']
    except (KeyError, IndexError):
        continue
    if not has_repair_signal(user_t2):
        continue
    n_repair_signal += 1
    m = REPAIR_RE.search(user_t2)
    if m:
        phrase_counter[m.group(0).lower()] += 1
    candidates.append({
        'conv_id': row.get('conversation_hash', ''),
        'user_t1': user_t1[:1500],
        'asst_t1': asst_t1[:1500],
        'user_t2': user_t2[:1500],
        'matched_phrase': m.group(0) if m else '',
    })

print(f'Total streamed:      {n_total}')
print(f'English:             {n_english}')
print(f'>= 4 turns:          {n_long_enough}')
print(f'Repair signal hit:   {n_repair_signal}')
print(f'Rate of repair signal among >=4-turn English: '
      f'{n_repair_signal / max(n_long_enough, 1) * 100:.3f}%')

## Matched-phrase distribution

In [ ]:
for phrase, count in phrase_counter.most_common(15):
    print(f'{count:>4}   {phrase}')

## Serialize

In [ ]:
funnel = {
    'sample_size': n_total,
    'english': n_english,
    'at_least_4_turns': n_long_enough,
    'repair_signal_hit': n_repair_signal,
    'rate_per_4turn_english': round(n_repair_signal / max(n_long_enough, 1) * 100, 3),
    'top_phrases': phrase_counter.most_common(20),
    'source': 'allenai/WildChat-1M (streamed sample)',
    'stage': 'A (regex only) -- no LLM judge yet',
}

with open('pilot_funnel.json', 'w') as f:
    json.dump(funnel, f, indent=2)

with open('pilot_examples.jsonl', 'w') as f:
    for c in candidates[:30]:
        f.write(json.dumps(c) + '\n')